# Session 1 — key check (run before class, ~10 min)

Four cells. If the last one prints **READY**, you are done. If it does not, you have
until the session to fix it — in class it will cost you your lab time, and **the lab has
no offline mode**: without a working key there is nothing to measure.

**The key is provided, and it is credited.** You do not sign up for anything and you
do not pay for anything. One key reaches all three models the lab compares — one from
Google, one from Z.ai, one open-weights model nobody here is hosting. Sign-ups before
the session were the single biggest reason a pair spent the first half hour of the last
run not measuring anything, so there are none.

If you do not have a key, or the check below does not print **READY**, email the
instructor **before** the session.

In [ ]:
# Cell 1/4 — install and configure
!pip install -q openai

import time
from getpass import getpass
from openai import OpenAI

# These are the lab notebook's own ids, written by build_docs.py from its `models` cell.
BASE_URL = "https://openrouter.ai/api/v1"
MODELS = {
    'gemini'       : 'google/gemini-3.5-flash-lite',
    'zai'          : 'z-ai/glm-5.3-flash',
    'deepseek'     : 'deepseek/deepseek-chat-v3.1',
}

print("One endpoint, three model ids. That is the whole configuration:")
for name, model_id in MODELS.items():
    print(f"  {name:12s} {model_id}")

In [ ]:
# Cell 2/4 — enter your key
key = getpass("OpenRouter API key: ")

client = OpenAI(api_key=key.strip(), base_url=BASE_URL, max_retries=5)

print("endpoint:", client.base_url)

In [ ]:
# Cell 3/4 — does each model actually answer?
results = {}
for name, model_id in MODELS.items():
    try:
        t0 = time.time()
        r = client.chat.completions.create(
            model=model_id, temperature=0,
            messages=[{"role": "user", "content": "Reply with exactly the word: ready"}],
        )
        dt = time.time() - t0
        results[name] = ("OK", f"{dt:.2f}s, {r.usage.prompt_tokens} in / "
                               f"{r.usage.completion_tokens} out")
    except Exception as e:
        results[name] = ("FAILED", f"{type(e).__name__}: {str(e)[:120]}")

for name in MODELS:
    status, detail = results[name]
    print(f"  {name:12s} {status:8s} {detail}")

In [ ]:
# Cell 4/4 — verdict
ok = [n for n, (s, _) in results.items() if s == "OK"]

print()
if len(ok) == 3:
    print("READY — all three models answer. See you in class.")
elif len(ok) >= 1:
    print(f"PARTIALLY READY — {len(ok)} of 3 models answer: {ok}")
    print("Your key works, so this is a model id or a routing problem, not a you")
    print("problem. Send the instructor the failing name and the error BEFORE the")
    print("session; the ids are confirmed a week ahead and one may have moved.")
else:
    print("NOT READY — nothing answered.")
    print("Re-read the error above. The usual cause is a key pasted with a stray")
    print("space or a missing character — try again first. If it still fails, the key")
    print("itself is the problem and that is the instructor's to fix, not yours.")
    print("Email BEFORE the session: this cannot be fixed in the first ten minutes of")
    print("class, and there is no offline mode to fall back on.")

# One number to bring with you, whatever happened above.
if ok:
    fastest = min(ok, key=lambda n: float(results[n][1].split("s,")[0]))
    print(f"\nFastest of your working providers on a trivial call: {fastest}")
    print("Write that down. In the lab you will find out whether it survives a real one.")